In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [5]:
import pandas as pd
from tqdm import tqdm
from utils import combined_approaches as ca
from utils import global_strategies as gs
from utils import label_based_measures as lbm
from utils import local_single_attribute as lsa
from utils import similarity_structures as ss
from utils import top_down_data_structures as tdds
from utils import value_overlap as vo
from utils import evaluation as eval
from utils import schema_integration as si
from utils import induced_match as im

<a id="esempio-bottomup-1"></a>
<a id="esempio-bottomup-1"></a>
<a id="esempio-bottomup-1"></a>
<a id="esempio-bottomup-1"></a>
<a id="esempio-bottomup-1"></a>
<a id="esempio-bottomup-1"></a>
<a id="esempio-bottomup-1"></a>
#  esempio BottomUp 1

In [3]:
src_links = [
'http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/BottomUp_2/S1.csv',
'http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/BottomUp_2/S2.csv'
]

SOURCES = { 'S'+str(i+1) : pd.read_csv(link).astype(str) for i, link in enumerate(src_links) }

GoldStandard=pd.read_csv('http://dbgroup.ing.unimore.it/SIWS/DataIntegration/Esempi/BottomUp_2/GoldStandard_A.csv').astype(str)

tdds.to_GMM(GoldStandard)

SOURCE,S1,S2
GAT,,
1,"[location, addr]","[location, indirizzo]"
2,[city],[city]
3,"[name_type, name]","[nome, descrizione]"
4,[telefone],[telefono]
5,[],[type]


In [4]:
SOURCES['S2']

,nome,city,telefono,type,location,descrizione,indirizzo
0,les celebrites,new york city,212-484-5113,french (classic),155 w. 58th st. (212-484-5113),les celebrites,155 w. 58th st. (212-484-5113)
1,delectables,atlanta,404-681-2909,cafeterias,1 margaret mitchell sq.,delectables,1 margaret mitchell sq. (404-681-2909)
2,lutece,new york city,212-752-2225,french (classic),249 e. 50th st.,lutece,249 e. 50th st. (212-752-2225)
3,georgia grille,atlanta,404-352-3517,southwestern,2290 peachtree rd. (404-352-3517),georgia grille,2290 peachtree rd. (404-352-3517)
4,plumpjack cafe,san francisco,415-563-4755,american (new),3127 fillmore st. (415-563-4755),plumpjack cafe,3127 fillmore st. (415-563-4755)
5,yujean kangs,pasadena,818-585-0855,chinese,67 n. raymond ave. (pasadena),yujean kangs,67 n. raymond ave. (818-585-0855)
6,phnom penh cambodian restaurant,san francisco,415-775-5979,cambodian,631 larkin st. (415-775-5979),phnom penh cambodian restaurant,631 larkin st. (415-775-5979)
7,island spice,new york city,212-765-1737,caribbean,402 w. 44th st. (212-765-1737),island spice (caribbean),402 w. 44th st.
8,sallys cafe & bakery,san francisco,415-626-6006,american,300 de haro st. (415-626-6006),sallys cafe & bakery,300 de haro st. (415-626-6006)
9,dotties true blue cafe,san francisco,415-885-2767,diners,522 jones st. (san francisco),dotties true blue cafe,522 jones st. (415-885-2767)


In [6]:
def CalcoloMatchingTable(TableL:pd.DataFrame,TableR:pd.DataFrame):

# Matching Methods: Qui si definiscono i Base Matcher da utilizzare; se ne possono aggiungere altri    
        SimTableA = lbm.levenshtein_label_based_similarity(TableL, TableR)
        # SimTableB = lbm.jaro_label_based_similarity(TableL, TableR)
        # SimTableB = vo.value_overlap_sim(GlobalSchema, Sources[y])
        SimTableB = vo.value_overlap_simjoin_jaccard(TableL, TableR, 0.6)

# Combined Approaches: qui si definisce il combiner; se ne possono aggiungere altri    
        # SimTable = ca.avg_sim_table([SimTableA,SimTableC])
        # SimTable = ca.min_sim_table([SimTableA,SimTableB,SimTableC])
        SimTable = ca.max_sim_table([SimTableA,SimTableB])

        # Weighted-sum
        #SimTable = ca.Weighted_sum([SimTableA,SimTableB,SimTableC], [.3,.4,.3] )

        
# Generating Correspondences: dalla tabella di similarità alle corrispondenze

### Local Single Attribute Strategies:
        MatchTable= lsa.thresholding(SimTable, 0.3)

        #MatchTable = lsa.top_K(SimTable,2,'A')
        #MatchTable = lsa.top_K(MatchTable,2,'B')

        
### Global Mapping (si usano solo questi due metodi)
        # MatchTable = gs.stable_marriage(MatchTable)
        # MatchTable = gs.simmetric_best_match(MatchTable)

        return MatchTable

In [7]:
def CalcoloLocalMatchingTable(Sources):
    LocalMatchingTable = pd.DataFrame(columns=['SOURCE_A', 'LAT_A', 'SOURCE_B', 'LAT_B', 'SLAT_A', 'SLAT_B', 'sim'])

    for x in Sources.keys():
        for y in Sources.keys():
            if (x < y):  

                LocalMatchingTableXY = CalcoloMatchingTable(Sources[x], Sources[y])

                print(x,y,len(LocalMatchingTableXY), end=" -- ")
                # si ottiene LocalMatchingTableXY , si rinominano gli attributi
                LocalMatchingTableXY.columns = ['LAT_A', 'LAT_B', 'sim']
                # si aggiungono i 2 nomi delle Local Sources matchate
                LocalMatchingTableXY['SOURCE_A'] = x
                LocalMatchingTableXY['SOURCE_B'] = y
                # e anche i SLAT
                LocalMatchingTableXY['SLAT_A'] = LocalMatchingTableXY['SOURCE_A'] + '_' + LocalMatchingTableXY['LAT_A']
                LocalMatchingTableXY['SLAT_B'] = LocalMatchingTableXY['SOURCE_B'] + '_' + LocalMatchingTableXY['LAT_B']
                # per poi aggiungerlo alla LocalMatchingTable complessiva
                LocalMatchingTable = LocalMatchingTable.append(LocalMatchingTableXY, sort=True)
    return LocalMatchingTable[['SOURCE_A', 'LAT_A', 'SOURCE_B', 'LAT_B', 'SLAT_A', 'SLAT_B', 'sim']]

In [8]:
def SchemaIntegration(Sources):

    LMT = CalcoloLocalMatchingTable(Sources)[['SLAT_A','SLAT_B']]
    
    GMT = si.GMTComponentiConnessi(LMT,Sources)
    

    return tdds.to_GMM(GMT)

In [14]:
GMT_Calcolata = tdds.to_GMT(SchemaIntegration(SOURCES))
GMT_Calcolata

S1 S2 8 -- 

,GAT,SOURCE,LAT,SLAT
0,1,S1,city,S1_city
1,1,S2,city,S2_city
2,2,S1,telefone,S1_telefone
3,2,S1,location,S1_location
4,2,S2,telefono,S2_telefono
5,2,S2,descrizione,S2_descrizione
6,2,S2,location,S2_location
7,3,S1,name_type,S1_name_type
8,3,S1,name,S1_name
9,3,S2,nome,S2_nome


In [15]:
eval.Valuta(im.MatchIndottiGMT(GoldStandard),im.MatchIndottiGMT(GMT_Calcolata))

,MT,TP,FP,FN,P,R,F
0,11,5,6,5,0.4545,0.5,0.4762
